# Practical 9: Slot Filling using Recurrent Neural Networks

**Problem Statement:** Implement sequence labeling for Spoken Language Understanding using RNN/LSTM.

**Activities:**
1. Text preprocessing
2. Train sequence labeling model
3. Evaluate prediction accuracy

**Dataset:** ATIS (Airline Travel Information System) — utterances labeled with per-word slot tags in IOB format.

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import urllib.request
import tensorflow as tf
from tensorflow import keras

TRAIN_URL = 'https://raw.githubusercontent.com/yvchen/JointSLU/master/data/atis-2.train.w-intent.iob'
TEST_URL = 'https://raw.githubusercontent.com/yvchen/JointSLU/master/data/atis.test.w-intent.iob'

def load_iob(url):
    lines = urllib.request.urlopen(url).read().decode('utf-8').strip().split('\n')
    sentences, tag_sequences = [], []
    for line in lines:
        text, labels = line.split('\t')
        tokens = text.split()[1:-1]
        tags = labels.split()[1:-1]
        sentences.append(tokens)
        tag_sequences.append(tags)
    return sentences, tag_sequences

train_sentences, train_tags = load_iob(TRAIN_URL)
test_sentences, test_tags = load_iob(TEST_URL)

print("Train examples:", len(train_sentences))
print("Test examples:", len(test_sentences))
print("\nSample sentence:", train_sentences[0])
print("Sample tags:    ", train_tags[0])

## 2. Text Preprocessing

Words and tags are mapped to integer indices, sequences are padded to a common length, and a padding mask is applied so the padded positions do not affect training or evaluation.

In [ ]:
words = sorted({w for sent in train_sentences for w in sent})
tags = sorted({t for seq in train_tags for t in seq})

word2idx = {w: i + 2 for i, w in enumerate(words)}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

tag2idx = {t: i + 1 for i, t in enumerate(tags)}
tag2idx['<PAD>'] = 0
idx2tag = {i: t for t, i in tag2idx.items()}

MAX_LEN = max(len(s) for s in train_sentences)

def encode(sentences, tag_seqs):
    X = [[word2idx.get(w, word2idx['<UNK>']) for w in s] for s in sentences]
    y = [[tag2idx.get(t, tag2idx['<PAD>']) for t in seq] for seq in tag_seqs]
    X = keras.preprocessing.sequence.pad_sequences(X, maxlen=MAX_LEN, padding='post')
    y = keras.preprocessing.sequence.pad_sequences(y, maxlen=MAX_LEN, padding='post')
    return X, y

X_train, y_train = encode(train_sentences, train_tags)
X_test, y_test = encode(test_sentences, test_tags)

print("Vocabulary size:", len(word2idx))
print("Number of tags:", len(tag2idx))
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 3. Sequence Labeling Model

A Bidirectional LSTM reads the full sentence in both directions, and a Dense layer applied at every time step predicts the slot tag for each word.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN,)),
    keras.layers.Embedding(input_dim=len(word2idx), output_dim=64, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True)),
    keras.layers.TimeDistributed(keras.layers.Dense(len(tag2idx), activation='softmax'))
])

model.summary()

## 4. Model Training

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=32,
    verbose=1
)

## 5. Evaluate Prediction Accuracy

Accuracy is computed only over real word positions, excluding padding.

In [ ]:
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=-1)

mask = y_test != tag2idx['<PAD>']
token_accuracy = (y_pred[mask] == y_test[mask]).mean()
print(f"Token-level accuracy (excluding padding): {token_accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 6. Sample Predictions

In [ ]:
for i in range(3):
    sent_len = len(test_sentences[i])
    print("Sentence:", ' '.join(test_sentences[i]))
    print("True tags:     ", test_tags[i])
    print("Predicted tags:", [idx2tag[idx] for idx in y_pred[i][:sent_len]])
    print()

## Conclusion

In this practical, we:
- Preprocessed the ATIS dataset into padded word and tag index sequences
- Built a Bidirectional LSTM sequence labeling model with masking
- Trained the model to predict per-word slot tags
- Evaluated token-level prediction accuracy and inspected sample predictions